# 005 - SQL Runner with CodeMirror · 데모

**005 의 popup 자동완성 + Python 콜백 실행** 을 한 번에 — CodeMirror 5.65.16 을 인라인 임베드해 "진짜 IDE" 같은 SQL 편집 경험.

## 005 / 006 / 007 한 눈에 비교

| 항목 | 005 (HTML/JS only) | **006 (CodeMirror 노트북)** | 007 (Textual TUI) |
|---|---|---|---|
| inline popup 자동완성 | ✅ | ✅ Ctrl+Space + 자동 popup | ✅ 인라인 OptionList (Tab) |
| 컨텍스트 추천 칩 패널 | ✅ | ✅ | ✅ |
| 커서 위치 정밀 인서트 | ✅ | ✅ 정확히 커서 위치 | ✅ |
| 에디터 자체 syntax 색 | ❌ | ✅ 에디터 내부 컬러링 | ✅ Textual native |
| line number / 라인 wrap | ❌ | ✅ | ✅ |
| ▶ 실행 → Python 콜백 | ❌ | ✅ | ✅ Ctrl+R |
| 결과 자동 표 렌더 | ❌ | ✅ pandas HTML | ✅ DataTable |
| 단축키 실행 | ❌ | ✅ Cmd/Ctrl+Enter | ✅ Ctrl+R / F5 |
| 의존성 | IPython | ipywidgets+IPython+CM(~285KB) | textual+rich (~25KB) |

## 시도할 것

- 에디터에 입력 시 `Ctrl+Space` (또는 식별자 입력 중 자동 popup) 으로 컨텍스트 인식 자동완성
- `WHE`, `GR`, `JOI` 같은 부분 키워드도 어느 컨텍스트에서나 popup 에 떠야 함 (fallback 적용)
- `users.` 처럼 `테이블명.` 입력 → 해당 테이블 컬럼만 한정 추천
- 좌측 entity 트리 / 하단 추천 칩 클릭 → CM 의 **현재 커서 위치** 에 정확히 인서트
- **Cmd/Ctrl + Enter** 로 ▶ 실행 → 아래 Output 영역에 DataFrame 의 **모든 컬럼** 까지 잘리지 않고 표 출력

> ⚠ **Trusted notebook 필요**: JupyterLab 에서 처음 열면 "Untrusted" 표시될 수 있고, 이 경우 인라인 `<script>` 가 차단됩니다. 메뉴 → File → Trust Notebook 로 trust 해주세요.

---

## 1️⃣ 임시 SQLite DB + 풍부한 데모 스키마

테이블 4개 (`users` · `orders` · `products` · `events`) — 컬럼은 6~8개씩, 다양한 타입 (`INTEGER` / `TEXT` / `REAL` / `TIMESTAMP` / `BOOLEAN`) 과 컬럼 description 포함.

In [ ]:
import os, sys, sqlite3, tempfile
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from sql_codemirror import SQLRunnerCM

db_path = os.path.join(tempfile.gettempdir(), 'sql_codemirror_demo.db')
if os.path.exists(db_path):
    os.remove(db_path)
with sqlite3.connect(db_path) as conn:
    conn.executescript('''
    CREATE TABLE users (
        id         INTEGER PRIMARY KEY,
        name       TEXT NOT NULL,
        email      TEXT,
        region     TEXT,
        plan_type  TEXT,
        is_active  INTEGER,
        signup_at  TIMESTAMP
    );
    CREATE TABLE orders (
        id              INTEGER PRIMARY KEY,
        user_id         INTEGER REFERENCES users(id),
        product_id      INTEGER REFERENCES products(id),
        quantity        INTEGER,
        amount          REAL,
        status          TEXT,
        payment_method  TEXT,
        ordered_at      TIMESTAMP
    );
    CREATE TABLE products (
        id          INTEGER PRIMARY KEY,
        sku         TEXT UNIQUE,
        name        TEXT NOT NULL,
        category    TEXT,
        price       REAL,
        stock       INTEGER,
        is_listed   INTEGER
    );
    CREATE TABLE events (
        id          INTEGER PRIMARY KEY,
        user_id     INTEGER REFERENCES users(id),
        event_type  TEXT,
        value       REAL,
        session_id  TEXT,
        occurred_at TIMESTAMP
    );

    INSERT INTO users (name, email, region, plan_type, is_active, signup_at) VALUES
        ('김알리스','alice@ex.com','서울','pro',     1,'2024-01-15 10:00'),
        ('이밥',   'bob@ex.com',  '부산','free',    1,'2024-02-20 12:30'),
        ('박찰리','charlie@ex.com','대구','pro',    0,'2024-03-10 09:15'),
        ('최다나','dana@ex.com', '서울','enterprise',1,'2024-04-01 14:45'),
        ('정에반','evan@ex.com', '인천','free',    1,'2024-05-22 18:00');

    INSERT INTO products (sku, name, category, price, stock, is_listed) VALUES
        ('SKU-A1','노트북 스탠드', 'office',     39000, 120, 1),
        ('SKU-B2','USB-C 허브',   'electronics',29000,  35, 1),
        ('SKU-C3','커피 원두 1kg','food',       18900, 200, 1),
        ('SKU-D4','요가 매트',    'fitness',    25000,  78, 1),
        ('SKU-E5','블루투스 키보드','electronics',88000, 12, 0);

    INSERT INTO orders (user_id, product_id, quantity, amount, status, payment_method, ordered_at) VALUES
        (1, 1, 1, 39000, 'paid',     'card',  '2024-05-01 11:00'),
        (1, 3, 2, 37800, 'paid',     'card',  '2024-05-08 09:20'),
        (2, 2, 1, 29000, 'paid',     'kakao', '2024-05-12 16:40'),
        (3, 4, 1, 25000, 'cancelled','card',  '2024-05-15 10:00'),
        (1, 5, 1, 88000, 'paid',     'naver', '2024-06-02 14:30'),
        (4, 1, 3,117000, 'paid',     'card',  '2024-06-10 11:15'),
        (2, 3, 1, 18900, 'pending',  'kakao', '2024-06-15 17:00'),
        (5, 4, 2, 50000, 'paid',     'card',  '2024-06-20 08:30');

    INSERT INTO events (user_id, event_type, value, session_id, occurred_at) VALUES
        (1, 'view',     NULL,    's1', '2024-06-01 10:00'),
        (1, 'click',    NULL,    's1', '2024-06-01 10:01'),
        (1, 'purchase', 39000,   's1', '2024-06-01 10:05'),
        (2, 'view',     NULL,    's2', '2024-06-02 11:00'),
        (3, 'view',     NULL,    's3', '2024-06-03 12:00'),
        (4, 'purchase', 117000,  's4', '2024-06-10 11:15'),
        (5, 'view',     NULL,    's5', '2024-06-15 09:00');
    ''')
print(f'✓ DB 준비 완료: {db_path}')
print(f'  4개 테이블, 총 {5+5+8+7} 행 데이터')

---

## 2️⃣ 풍부한 컬럼 description / 타입 정보로 위젯 띄우기

`from_sqlite` 로 자동 추출되는 정보 위에 **컬럼별 doc** 을 한국어로 보강 — 트리 컬럼 hover 시 tooltip 으로 노출, 자동완성 popup 의 `displayText` 에도 반영.

In [ ]:
runner = SQLRunnerCM.with_sqlite(db_path)

# 자동 추출된 type 위에 한국어 description 덧붙이기
runner.add_table('users', [
    {'name': 'id',        'type': 'INTEGER',   'doc': 'PK · 사용자 고유번호'},
    {'name': 'name',      'type': 'TEXT',      'doc': '표시 이름 (실명 · 닉네임)'},
    {'name': 'email',     'type': 'TEXT',      'doc': '로그인용 이메일'},
    {'name': 'region',    'type': 'TEXT',      'doc': '거주 지역 (서울·부산·...)'},
    {'name': 'plan_type', 'type': 'TEXT',      'doc': 'free / pro / enterprise'},
    {'name': 'is_active', 'type': 'INTEGER',   'doc': '활성 여부 (1/0 · 0=탈퇴)'},
    {'name': 'signup_at', 'type': 'TIMESTAMP', 'doc': '가입 시각 (UTC)'},
], description='사용자 마스터 (5명)')
runner.add_table('orders', [
    {'name': 'id',             'type': 'INTEGER',   'doc': 'PK'},
    {'name': 'user_id',        'type': 'INTEGER',   'doc': 'FK → users.id'},
    {'name': 'product_id',     'type': 'INTEGER',   'doc': 'FK → products.id'},
    {'name': 'quantity',       'type': 'INTEGER',   'doc': '구매 수량'},
    {'name': 'amount',         'type': 'REAL',      'doc': '결제 금액 (KRW · 할인 후)'},
    {'name': 'status',         'type': 'TEXT',      'doc': 'paid / pending / cancelled'},
    {'name': 'payment_method', 'type': 'TEXT',      'doc': 'card / kakao / naver / ...'},
    {'name': 'ordered_at',     'type': 'TIMESTAMP', 'doc': '주문 시각'},
], description='주문 내역 (8건)')
runner.add_table('products', [
    {'name': 'id',         'type': 'INTEGER', 'doc': 'PK'},
    {'name': 'sku',        'type': 'TEXT',    'doc': '상품 SKU (UNIQUE)'},
    {'name': 'name',       'type': 'TEXT',    'doc': '상품명'},
    {'name': 'category',   'type': 'TEXT',    'doc': 'office/electronics/food/fitness'},
    {'name': 'price',      'type': 'REAL',    'doc': '단가 (KRW · 정가)'},
    {'name': 'stock',      'type': 'INTEGER', 'doc': '현재 재고'},
    {'name': 'is_listed',  'type': 'INTEGER', 'doc': '노출 여부 (1/0)'},
], description='상품 카탈로그 (5종)')
runner.add_table('events', [
    {'name': 'id',          'type': 'INTEGER',   'doc': 'PK'},
    {'name': 'user_id',     'type': 'INTEGER',   'doc': 'FK → users.id'},
    {'name': 'event_type',  'type': 'TEXT',      'doc': 'view / click / purchase / ...'},
    {'name': 'value',       'type': 'REAL',      'doc': 'event-specific 수치 (purchase=금액)'},
    {'name': 'session_id',  'type': 'TEXT',      'doc': 'UA 세션 식별자'},
    {'name': 'occurred_at', 'type': 'TIMESTAMP', 'doc': '이벤트 발생 시각'},
], description='유저 행동 로그 (7건)')

runner.set_query(
    "-- 사용자별 결제 합계 + 평균 + 카드 결제 비율 + 마지막 주문일\n"
    "SELECT\n"
    "    u.name,\n"
    "    u.region,\n"
    "    u.plan_type,\n"
    "    COUNT(o.id)                              AS n_orders,\n"
    "    SUM(CASE WHEN o.status='paid' THEN o.amount ELSE 0 END) AS paid_total,\n"
    "    AVG(CASE WHEN o.status='paid' THEN o.amount END)        AS paid_avg,\n"
    "    SUM(CASE WHEN o.payment_method='card' THEN 1 ELSE 0 END) * 1.0 / COUNT(o.id) AS card_ratio,\n"
    "    MAX(o.ordered_at)                        AS last_ordered_at\n"
    "FROM users u\n"
    "LEFT JOIN orders o ON o.user_id = u.id\n"
    "GROUP BY u.id\n"
    "ORDER BY paid_total DESC NULLS LAST;"
)
runner.show()

**시도해보세요**:

1. 에디터에서 `Cmd/Ctrl + Enter` → 위 8-열 결과가 잘림 없이 모든 컬럼 표 출력
2. 새 줄에 `SELECT product_id, SUM(amount) FROM orders WHE` 까지 입력 → `WHERE` 가 popup 에 자동으로 뜸 (fallback 키워드)
3. `users.` / `orders.` / `products.` 까지만 입력 → 해당 테이블의 컬럼만 한정 추천 + tooltip 에 한국어 doc 표시
4. 에디터 아래 **💡 추천** 칩 클릭 → CM 의 현재 커서 위치에 정확히 인서트 (006 처럼 끝에 append 되지 않음)
5. 좌측 컬럼 버튼 hover → 한국어 description 이 tooltip 으로 표시

---

## 3️⃣ 다양한 SQL 백엔드 연동 — sqlite 외 사례

`SQLRunnerCM(on_execute=fn)` 의 `fn` 은 **임의 함수** 면 됩니다. 시그니처는 `(sql: str) -> Any`. 대표 패턴 4가지:

### 3-1. SQLite raw cursor — pandas 없이 list[dict] 반환

결과를 `list[dict]` 로 반환하면 위젯이 자동으로 `pd.DataFrame` 으로 변환해 표 렌더 (pandas 가 있을 때).

In [ ]:
def run_sql_raw(sql: str):
    """sqlite3 만 사용 — pandas 미의존. 결과는 list[dict]."""
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(sql).fetchall()
        return [dict(r) for r in rows]

raw_runner = SQLRunnerCM(on_execute=run_sql_raw)
raw_runner.from_sqlite(db_path)
raw_runner.set_query('SELECT id, name, region, plan_type FROM users WHERE is_active = 1;')
raw_runner.show()

### 3-2. 메모리 내 mock 엔진 — DB 없이 Python dict 위에서 동작

사내 시스템·캐시·테스트 더블 처럼 **DB 가 없는 환경** 에서도 위젯은 그대로 사용 가능. 콜백이 SQL 문자열을 자기 식으로 해석만 하면 됨.

In [ ]:
MEMORY = {
    'users':    [{'id': 1, 'name': '김알리스', 'region': '서울'},
                 {'id': 2, 'name': '이밥',     'region': '부산'},
                 {'id': 3, 'name': '박찰리',   'region': '대구'}],
    'products': [{'id': 'A1', 'name': '노트북 스탠드', 'price': 39000},
                 {'id': 'B2', 'name': 'USB-C 허브',   'price': 29000}],
}

def mock_engine(sql: str):
    """매우 단순한 mock — `SELECT * FROM <table>` 만 인식.
    실제 사내 환경에서는 이 자리에 자체 query parser / API client 로 교체."""
    import re
    m = re.search(r'from\s+(\w+)', sql, re.I)
    if not m:
        return f'⚠ 데모 mock 은 SELECT ... FROM <table> 만 지원합니다.\n받은 SQL:\n{sql}'
    name = m.group(1)
    if name not in MEMORY:
        return f'⚠ 알 수 없는 테이블: {name}'
    return MEMORY[name]   # list[dict] → 자동 표 렌더

mock_runner = SQLRunnerCM(on_execute=mock_engine)
mock_runner.from_dict({
    'users':    [{'name': 'id', 'type': 'INT', 'doc': ''},
                 {'name': 'name', 'type': 'TEXT', 'doc': ''},
                 {'name': 'region', 'type': 'TEXT', 'doc': ''}],
    'products': [{'name': 'id', 'type': 'TEXT', 'doc': ''},
                 {'name': 'name', 'type': 'TEXT', 'doc': ''},
                 {'name': 'price', 'type': 'INT', 'doc': ''}],
})
mock_runner.set_query('SELECT * FROM users;')
mock_runner.show()

### 3-3. pandas DataFrame 위에서 직접 실행 — `df.query` 패턴

이미 메모리에 로드된 DataFrame 들을 SQL-스러운 인터페이스로 다루는 시나리오. 콜백을 `df.query` / `pandasql` / `duckdb.sql` 등으로 갈아끼우면 됨.

In [ ]:
df_users = pd.read_sql('SELECT * FROM users', sqlite3.connect(db_path))
df_orders = pd.read_sql('SELECT * FROM orders', sqlite3.connect(db_path))
DFS = {'users': df_users, 'orders': df_orders}

def df_engine(sql: str):
    """WHERE 만 지원하는 단순 데모 엔진 — `SELECT ... FROM <table> [WHERE <expr>]` 인식.
    실제로는 pandasql.sqldf / duckdb.sql(...).df() 로 교체하면 즉시 사용 가능."""
    import re
    m = re.search(r'from\s+(\w+)\s*(?:where\s+(.+?))?(?:;|$)',
                  sql.strip().rstrip(';'), re.I | re.S)
    if not m or m.group(1) not in DFS:
        return f'⚠ 데모 엔진 한계 — 받은 SQL:\n{sql}'
    df = DFS[m.group(1)]
    expr = m.group(2)
    if expr:
        try:
            df = df.query(expr.strip())   # `is_active == 1` 등 pandas expr
        except Exception as e:
            return f'❌ df.query 실패: {e}\n표현식: {expr!r}'
    return df

df_runner = SQLRunnerCM(on_execute=df_engine)
df_runner.from_dataframes({'users': df_users, 'orders': df_orders})
df_runner.set_query('SELECT * FROM users WHERE is_active == 1 and region == "서울";')
df_runner.show()

### 3-4. 콜백 echo — 진짜 엔진 연결 전 디자인 확인용

`on_execute` 가 실제 엔진 호출 없이 SQL 만 echo. 위젯 UX 자체를 점검할 때 유용.

In [ ]:
def echo_only(sql):
    print('=== 받은 SQL ===')
    print(sql)
    print()
    print('=== 사내 엔진 (REST / Spark / Snowflake / 자체 query 엔진 ...) 과 ===')
    print('=== 연결되면 결과가 여기에 출력됩니다 ===')

echo = SQLRunnerCM(on_execute=echo_only)
echo.from_dict({
    'customers': [
        {'name': 'id',      'type': 'INT',  'doc': 'PK'},
        {'name': 'name',    'type': 'TEXT', 'doc': '고객명'},
        {'name': 'phone',   'type': 'TEXT', 'doc': '전화'},
    ],
    'invoices': [
        {'name': 'id',          'type': 'INT',  'doc': 'PK'},
        {'name': 'customer_id', 'type': 'INT',  'doc': 'FK'},
        {'name': 'total',       'type': 'REAL', 'doc': '청구 합계'},
    ],
})
echo.set_query('SELECT * FROM customers WHERE name LIKE "%김%";')
echo.show()

---

## 4️⃣ 추천 함수 단위 테스트 (위젯 없이 직접 호출)

위젯 외부에서도 컨텍스트 감지 / 추천 로직만 떼어 사용 가능 — 사내 다른 도구 (코드 검사기 · IDE 플러그인 등) 에 임베드할 때 유용.

In [ ]:
from sql_codemirror import detect_context, get_suggestions

schema = {
    'users':  [{'name': 'id', 'type': 'INT', 'doc': ''},
               {'name': 'name', 'type': 'TEXT', 'doc': ''},
               {'name': 'region', 'type': 'TEXT', 'doc': ''}],
    'orders': [{'name': 'id', 'type': 'INT', 'doc': ''},
               {'name': 'user_id', 'type': 'INT', 'doc': ''},
               {'name': 'amount', 'type': 'REAL', 'doc': ''}],
}
queries = [
    'SELECT ',
    'SELECT * FROM ',
    'SELECT * FROM users WHE',          # ← fallback 키워드 매칭
    'SELECT * FROM users GR',           # ← fallback 키워드 매칭
    'SELECT * FROM orders WHERE ',
    'SELECT * FROM orders WHERE orders.',
    'SELECT name FROM users INNER ',    # ← join_continue
]
for q in queries:
    ctx = detect_context(q)
    sugs = get_suggestions(q, schema)
    top5 = ', '.join(s['label'] for s in sugs[:5])
    print(f'  {q!r:42}  →  ctx={ctx:18}  top5: {top5}')

---

## 5️⃣ 대량 스키마 stress — 100+ 테이블 (사내 금융사급)

좌측 entity 패널은 위젯 객체(W.Button) 를 만들지 않고 단일 HTML 한 덩어리로 렌더하므로 수백~수천 개 컬럼이 있는 사내 스키마도 첫 렌더 지연이 거의 없습니다. 검색창에 입력하면 **JS 자체 필터**로 매치되는 테이블/컬럼만 즉석 표시 (Python round-trip 0).

아래 셀은 사내 금융사 시뮬 스키마 — **10개 도메인 × ~110 테이블 × 8~25 컬럼** 으로 약 **1,800+ entities** 를 만들어 패널 동작을 확인합니다.

**검색창 시도**:
- `customer` → 고객 도메인 11개 테이블만 필터
- `_id` → 모든 FK 컬럼만 필터
- `amount`, `balance` → 금액/잔액 컬럼만 매치
- `kyc` → 컴플라이언스 KYC 관련만

**자동완성 시도**:
- 에디터에서 `SELECT * FROM ` 입력 → popup 에 ~110 테이블 노출
- `SELECT * FROM portfolio_` → portfolio_* 테이블만 필터
- `SELECT * FROM customers WHERE ` → customers 컬럼만 매치

> 💾 **CSV/Excel 파일 저장** 버튼도 같이 시연됩니다 — `▶ 실행` 후 `💾 CSV 파일 저장` 을 누르면 노트북 cwd 에 `sql_result_<ts>.csv` 가 생기고 클릭 가능 링크가 출력됩니다.

In [ ]:
import random
random.seed(42)

def fake_engine(sql: str):
    """실제 엔진 없이 구색 결과만 반환 — 저장 버튼 시연용."""
    return [{"row_id": i, "value": f"val_{i:03d}", "score": i * 1.5}
            for i in range(20)]

# ── 사내 금융사 시뮬 스키마: 10 도메인 · 110+ 테이블 ──
SCHEMA_GROUPS = {
    "고객": [
        "customers", "customer_addresses", "customer_contacts",
        "customer_documents", "customer_consents", "customer_segments",
        "customer_relationships", "customer_notes", "kyc_records",
        "beneficial_owners", "customer_risk_profiles",
    ],
    "계좌": [
        "accounts", "account_holders", "account_balances",
        "account_statements", "account_transactions", "account_holds",
        "account_overdrafts", "account_fees", "account_limits",
        "account_types", "account_alerts", "savings_accounts",
        "checking_accounts", "credit_accounts", "investment_accounts",
    ],
    "거래": [
        "transactions", "trade_executions", "settlements", "clearings",
        "wire_transfers", "internal_transfers", "fee_transactions",
        "currency_exchanges", "dividend_payments", "interest_payments",
        "transaction_journal", "pending_transactions", "reversals",
    ],
    "주문": [
        "orders", "order_executions", "order_routing", "order_book",
        "order_states", "order_amendments", "order_cancellations",
        "basket_orders", "algorithmic_orders", "dark_pool_orders",
        "smart_routing_logs",
    ],
    "상품/증권": [
        "instruments", "equity_securities", "fixed_income_securities",
        "derivatives", "options", "futures", "swaps", "mutual_funds",
        "etfs", "structured_products", "instrument_master",
        "instrument_prices", "instrument_dividends", "corporate_actions",
    ],
    "포트폴리오": [
        "portfolios", "portfolio_holdings", "portfolio_allocations",
        "portfolio_returns", "portfolio_metrics", "portfolio_constraints",
        "portfolio_rebalances", "portfolio_risk", "portfolio_benchmarks",
    ],
    "컴플라이언스": [
        "aml_screenings", "sanctions_lists", "watchlists",
        "suspicious_activities", "compliance_alerts",
        "regulatory_filings", "audit_trails", "ofac_screenings",
        "pep_records", "compliance_cases",
    ],
    "분석/이벤트": [
        "events", "page_views", "sessions", "user_journeys",
        "funnel_steps", "conversion_events", "click_streams",
        "error_events", "performance_metrics", "ab_test_assignments",
    ],
    "참조데이터": [
        "currencies", "exchange_rates", "countries", "business_calendars",
        "holidays", "market_data_sources", "reference_indices",
        "sector_classifications", "gics_codes", "isin_database",
    ],
    "시스템": [
        "system_users", "system_roles", "system_permissions",
        "system_sessions", "api_keys", "audit_logs", "system_logs",
        "error_logs", "access_logs", "system_configuration",
    ],
}

# ── 컬럼 풀 (도메인 무관 공통) ──
NUMERIC_COLS = ["amount", "balance", "fee", "rate", "score",
                "quantity", "volume", "weight", "factor", "threshold",
                "premium", "discount", "tax", "interest_amount",
                "principal", "yield_rate", "spread", "ratio",
                "percentage", "market_value", "book_value", "nav",
                "price", "unit_cost"]
TEXT_COLS = ["name", "code", "description", "category", "type",
             "status", "label", "tag", "notes", "comment",
             "reason", "source", "channel", "method", "currency_code",
             "country_code", "language_code", "external_id",
             "reference_no", "approval_code"]
DATE_COLS = ["created_at", "updated_at", "deleted_at", "effective_date",
             "expiration_date", "settled_at", "executed_at",
             "approved_at", "submitted_at", "processed_at",
             "cancelled_at", "valued_at"]
INT_COLS = ["version", "sequence_no", "priority", "max_retries",
            "attempt_count", "row_count", "line_no"]

# 자동 FK 추론용 prefix
FK_PREFIXES = ["customer", "account", "order", "transaction",
               "instrument", "portfolio", "product", "user"]


def generate_columns(table_name: str, count: int) -> list:
    """테이블 이름에 맞는 컬럼 목록 생성 — id PK + 자동 FK + 랜덤 풀."""
    cols = [{"name": "id", "type": "INTEGER", "doc": "PK"}]
    used = {"id"}
    # 자동 FK — 테이블 이름이 이 prefix 로 시작하지 않으면서 단어로 포함
    for prefix in FK_PREFIXES:
        plural = f"{prefix}s"
        if (table_name.startswith(prefix) and table_name != prefix
                and table_name != plural):
            continue
        if prefix in table_name.split("_"):
            fk = f"{prefix}_id"
            if fk not in used:
                cols.append({"name": fk, "type": "INTEGER",
                             "doc": f"FK → {plural}.id"})
                used.add(fk)
    pools = [(NUMERIC_COLS, "REAL"), (TEXT_COLS, "TEXT"),
             (DATE_COLS, "TIMESTAMP"), (INT_COLS, "INTEGER")]
    while len(cols) < count:
        pool, ctype = random.choice(pools)
        candidate = random.choice(pool)
        if candidate not in used:
            used.add(candidate)
            doc = f"{candidate} ({ctype.lower()})"
            cols.append({"name": candidate, "type": ctype, "doc": doc})
    return cols


big = SQLRunnerCM(on_execute=fake_engine)
total_tables = 0
total_cols = 0
for group, tables in SCHEMA_GROUPS.items():
    for tname in tables:
        ncols = random.randint(8, 25)
        big.add_table(tname, generate_columns(tname, ncols),
                      description=f"[{group}] {tname.replace('_', ' ')}")
        total_tables += 1
        total_cols += ncols

print(f"✓ {len(SCHEMA_GROUPS)} 도메인 · {total_tables} 테이블 · "
      f"{total_cols} 컬럼 (총 {total_tables + total_cols} entities)")
print(f"  도메인: {', '.join(SCHEMA_GROUPS.keys())}")
big.set_query("-- 검색창에 customer / portfolio / kyc 등 입력 → 즉시 필터\n"
              "-- 자동완성: SELECT * FROM 입력 시 110+ 테이블 후보 노출\n"
              "SELECT * FROM customers LIMIT 20;")
big.show()


---

## 6️⃣ 실행 이력 — `runner.history()` + 클립보드 복사

▶ 실행 버튼을 누를 때마다 **timestamp · SQL · 결과 · 에러** 가 `runner.history` 에 자동 누적됩니다.

| 형태 | 설명 |
|---|---|
| `runner.history` | list 처럼 동작 — 인덱싱 / for-loop / append / `len(...)` 모두 OK |
| `runner.history()` | 노트북에 HTML 로 보기 좋게 표시 + **항목별 [📋 SQL 복사]** + 상단 **[📋 전체 history 복사 (markdown)]** |
| `runner.history(n=10)` | 최근 10건만 표시 |
| `runner.history(full=False)` | 결과 미리보기 생략 — SQL/timestamp 만 빠르게 훑기 |
| `runner.history.to_markdown()` | 직렬화 텍스트 (사내 위키 / Slack / 이슈 트래커 공유용) |

복사 동작은 `navigator.clipboard` 우선, 권한 차단 시 hidden textarea + `execCommand('copy')` 폴백 — **폐쇄망 / iframe** 환경에서도 동작합니다.

아래 셀은 데모용으로 5개 SQL (성공 4 + 에러 1) 을 즉석 실행해 history 를 채운 뒤 `runner.history()` 를 호출합니다. 실제 사용에서는 ▶ 실행 버튼이 자동으로 누적합니다.

In [ ]:
import datetime, time

# 위 cell 4 의 runner (4 테이블 sqlite 백엔드) 를 재사용 — 실제 사용에서는
# ▶ 실행 버튼이 자동으로 history 에 timestamp 와 함께 누적합니다.
runner.history.clear()

demo_queries = [
    "SELECT COUNT(*) AS total_users FROM users",
    "SELECT region, COUNT(*) AS n FROM users "
    "GROUP BY region ORDER BY n DESC",
    ("SELECT u.name, u.region, SUM(o.amount) AS total\n"
     "  FROM users u JOIN orders o ON o.user_id = u.id\n"
     "  WHERE o.status = 'paid'\n"
     "  GROUP BY u.id ORDER BY total DESC LIMIT 5"),
    "SELECT * FROM products WHERE price > 50000",
    "SELECT * FROM nonexistent_table",   # ← 의도적 에러 (백엔드 실패)
]

for q in demo_queries:
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    try:
        result = runner.on_execute(q)
        runner.history.append({
            "timestamp": ts, "query": q,
            "result": result, "error": None,
        })
    except Exception as e:
        runner.history.append({
            "timestamp": ts, "query": q,
            "result": None, "error": e,
        })
    time.sleep(0.6)   # timestamp 가 시각적으로 다르도록

ok = sum(1 for h in runner.history if h["error"] is None)
err = sum(1 for h in runner.history if h["error"] is not None)
print(f"✓ history 에 {len(runner.history)} 건 누적 (성공 {ok} · 에러 {err})\n")

# 📋 → runner.history() 호출 시 HTML 위에 [복사] 버튼이 함께 렌더됨
runner.history()

**markdown 직렬화도 가능** — 사내 위키나 이슈 트래커에 그대로 붙여넣을 때 유용:

In [ ]:
print(runner.history.to_markdown())

---

## 정리

- **에디터 내부 syntax highlight + popup 자동완성** — 005/006 가 따로 가졌던 것을 한 위젯에서 제공
- **하단 추천 칩** — popup 과 별개로 항상 보이는 컨텍스트 추천 (005 와 동일한 컨셉)
- **fallback 키워드** — `WHE`, `GR`, `JOI` 같은 부분 입력도 어느 컨텍스트에서나 매치
- **모든 컬럼 표시** — DataFrame 결과는 가로 스크롤로 모든 열을 잘리지 않게 표 렌더
- **임의 콜백 연동** — sqlite, list[dict], 메모리 dict, DataFrame, REST API, 사내 엔진 등 모두 가능
- **인라인 임베드** — `_assets/` 폴더 없이 `sql_codemirror.py` 한 파일만 폐쇄망에 반입하면 됨

## 폐쇄망 친화 체크

| 항목 | 상태 |
|---|---|
| 외부 네트워크 / CDN | ❌ 없음 (모든 CSS/JS 인라인) |
| `<link href>` / `<script src>` | ❌ 없음 |
| 새 서버 / 포트 | ❌ 없음 |
| 바이너리 영속화 | ❌ 없음 |
| 단일 반입 단위 | `sql_codemirror.py` 한 파일 |
| 추가 패키지 | ipywidgets + IPython (이미 스택 포함) |